# Grading Records

Your mark sheet for the whole course. Every attempt at every test is logged in
`records.json`, and this notebook turns that ledger into a progress report:
per-module best scores, the weighted course mark, the band, and the next
module to attack.

- `records.json` - the ledger (plain JSON, safe to commit, yours to edit).
- `grade_records.py` - the command line tool that writes it and reports on it.
- `GRADE_REPORT.md` - the exported markdown report, ready for your portfolio.

The cells below are live: they read the ledger every time you run them.

## 1. Recording an attempt

**After you have worked through a test in Jupyter**, grade it and log it in one
step:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py run "T03 - Methods and Functions Test.ipynb" --prose 7
```

The grader replays your saved notebook, counts the checks that pass, then asks
for the points for the hand-marked questions (`--prose`). Those are the
multiple-choice and conceptual questions: 1 point each.

**If you already know your numbers**, record them directly:

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py record T00 --checks 5/6 --prose 6 --notes "slicing still slips"
```

**Read it back**

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py list
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py summary
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py report
```

**Start a clean attempt for one module**

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py reset T03
```

Your best score per module is kept, so a retake can only improve your record.

## 2. Live report

Run the cell below to see the current state of your record.

In [ ]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(".")) if False else os.getcwd()
if not os.path.exists(os.path.join(HERE, "grade_records.py")):
    HERE = r"C:/Users/andre/Learning/Udemy-Complete-Python-3-Bootcamp/20-Testing-Syllabus"
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import grade_records as gr

try:
    from IPython.display import Markdown, display

    def show_md(text):
        display(Markdown(text))
except (ImportError, NameError):
    def show_md(text):
        print(text)

**Course mark and band**

In [ ]:
summary = gr.summarize()

lines = [
    "| Measure | Value |",
    "| --- | --- |",
    "| Course mark | **%.1f / 100** (%s) |" % (summary["course_mark"] * 100, summary["band"]),
    "| Section tests | %.1f / 100  (85%% of the mark) |" % (summary["section_mark"] * 100),
    "| Final exam | %s  (15%% of the mark, pass %d/60) |" % (
        ("%.1f / 100" % (summary["final_mark"] * 100)) if summary["final_points"] is not None else "not attempted",
        gr.FINAL_PASS),
    "| Modules signed off | %d / %d (pass mark %d/20) |" % (
        len(summary["signed_off"]), len(summary["rows"]), gr.PASS_MARK),
    "| Attempts logged | %d |" % len(summary["attempts"]),
]
if summary["next_target"]:
    lines.append("| Next target | %s |" % summary["next_target"])

show_md("\n".join(lines))

**Per-module best scores**

In [ ]:
rows = ["| Test | Module | Best | Weight | Progress |", "| --- | --- | --- | --- | --- |"]
for row in summary["rows"]:
    test = row["test"]
    best = row["best"]
    mark = ("%d / %d" % (best, test["points_possible"])) if best is not None else "- / %d" % test["points_possible"]
    state = "signed off" if best is not None and best >= gr.PASS_MARK else (
        "attempted" if best is not None else "not attempted")
    rows.append("| %s | %s | %s | %d%% | %s %s |" % (
        test["id"], test["title"], mark, test["weight"], gr.format_bar(best, test["points_possible"]), state))

show_md("\n".join(rows))

**Attempt history**

In [ ]:
if not summary["attempts"]:
    show_md("No attempts recorded yet. Run `grade_records.py run ...` after your first test.")
else:
    rows = ["| Date | Test | Checks | Coding | Prose | Total | Notes |", "| --- | --- | --- | --- | --- | --- | --- |"]
    for attempt in summary["attempts"]:
        rows.append("| %s | %s | %d/%d | %d | %d | **%d / %d** | %s |" % (
            attempt["date"], attempt["test"], attempt["checks_passed"], attempt["checks_total"],
            attempt["coding_points"], attempt["prose_points"], attempt["points"],
            attempt["points_possible"], attempt["notes"] or "-"))
    show_md("\n".join(rows))

**Sign-off checklist**

A module is signed off at 14 / 20. Tick them here, or in the readiness
checklist at the end of `00 - Testing Syllabus.ipynb`.

In [ ]:
rows = []
for row in summary["rows"]:
    test = row["test"]
    best = row["best"]
    done = best is not None and best >= gr.PASS_MARK
    rows.append("- [%s] **%s** %s - %d/%d%s" % (
        "x" if done else " ", test["id"], test["title"], best if best is not None else 0,
        test["points_possible"], " (signed off)" if done else ""))
rows.append("")
if summary["final_points"] is not None:
    final_done = summary["final_points"] >= gr.FINAL_PASS
    rows.append("- [%s] **99** Final Comprehensive Exam - %d/60%s" % (
        "x" if final_done else " ", summary["final_points"], " (signed off)" if final_done else ""))
else:
    rows.append("- [ ] **99** Final Comprehensive Exam - not attempted")

show_md("\n".join(rows))

## 3. How the course mark is calculated

Each module test is worth 20 points and carries a weight from the blueprint in
`00 - Testing Syllabus.ipynb`. Your section mark is the weighted average of
your **best** attempt at each module:

```
section mark = sum(best_points * weight) / sum(20 * weight)      -> 85% of the course mark
final exam   = best_points / 60                                  -> 15% of the course mark
course mark  = 0.85 * section mark + 0.15 * final exam
```

Bands: Distinction 85+, Merit 70+, Pass 55+, below pass under 55.

**Why the best attempt counts.** Retakes are how learning works here. The
ledger keeps every attempt so you can see the trend, but only your best score
per module counts towards the mark.

## 4. The ledger file

`records.json` holds one entry per attempt:

```json
{
  "attempts": [
    {
      "date": "2026-11-13",
      "test": "T03",
      "title": "Methods and Functions",
      "checks_passed": 6,
      "checks_total": 6,
      "coding_points": 12,
      "prose_points": 7,
      "points": 19,
      "points_possible": 20,
      "notes": "kwargs finally clicked"
    }
  ]
}
```

It is plain JSON: read it, edit it, or keep it in a spreadsheet tool. The
notebook and the CLI both read the same file, so they always agree.

## 5. Using the record as portfolio evidence

```
C:\Users\andre\anaconda3\python.exe 20-Testing-Syllabus/grade_records.py report
```

writes `GRADE_REPORT.md`: the course mark, the per-module table, the attempt
history and the signed-off checklist. Commit it to your portfolio repository
under `docs/` and it becomes a dated record of your progress through the
course - the same pattern described in `03 - Project Portfolio.ipynb`.